# Test notebook to exercise the Lasair API
### To run this notebook, please [follow the instructions](https://lasair-lsst.readthedocs.io/en/main/core_functions/python-notebooks.html) or else it won`t work.
The instructions are at https://lasair-lsst.readthedocs.io/en/main/core_functions/python-notebooks.html

In [11]:
import sys
lsst=True # if you change this restart kernel

if lsst:
    sys.path.append('API_lsst')
    import settings
    endpoint = "https://api.lasair.lsst.ac.uk/api"
    oid = 'diaObjectId'
    raname = 'ra'
    decname = 'decl'
else:
    sys.path.append('API_ztf')
    import settings
    endpoint = "https://lasair-ztf.lsst.ac.uk/api"
    oid = 'objectId'
    raname = 'ramean'
    decname = 'decmean'

In [12]:
import json
from lasair import LasairError, lasair_client as lasair
import settings
L = lasair(settings.API_TOKEN, endpoint=endpoint)
help(lasair)

Help on class lasair_client in module lasair.lasair:

class lasair_client(builtins.object)
 |  lasair_client(token, cache=None, endpoint='https://lasair-ztf.lsst.ac.uk/api', timeout=60.0)
 |
 |  Methods defined here:
 |
 |  __init__(self, token, cache=None, endpoint='https://lasair-ztf.lsst.ac.uk/api', timeout=60.0)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  annotate(self, topic, objectId, classification, version='0.1', explanation='', classdict={}, url='')
 |      Send an annotation to Lasair
 |      args:
 |          topic         : the topic for which this user is authenticated
 |          objectId      : the object that this annotation should be attached to
 |          classification: short string for the classification
 |          version       : the version of the annotation engine
 |          explanation   : natural language explanation
 |          classdict     : dictionary with further information
 |          url           : url with further 

This function simply prints the first lines of the given string.
You can replace it with the normal print if you want to see everything.

In [13]:
def print15(s):
    print('\n'.join(s.split('\n')[:15]))

## Cone search

In [14]:
racone  = 54.869
deccone = -27.294
radius = 60 # arcsec
result = L.cone(racone, deccone, radius, requestType='nearest')
print(json.dumps(result, indent=2))

{
  "nearest": {
    "object": 313774267128873067,
    "separation": 57.69581417645488
  }
}


## Database query

In [15]:
selected = 'objects.%s, objects.%s, objects.%s' % (oid, raname, decname)
tables = 'objects,sherlock_classifications'
conditions = 'classification="SN"'
results = L.query(selected, tables, conditions, limit = 8)

for row in results:
    objectId = row[oid]
    ra = row[raname]
    dec = row[decname]
    print(objectId, ra, dec)

# Just use this objectId in subsequent calls
objectId = str(objectId)

170019696254910754 62.00489102706123 -49.480107426965155
170019696255435601 61.968117601750194 -49.66424587878178
170019696256483349 62.03430336082304 -49.40208365299349
170019696256483399 62.080921093801514 -49.282687331866626
170019696256483574 62.019254932653006 -49.237951802496646
170019696262250510 60.76850579249797 -50.18300078916755
170019696266444924 59.8990394301128 -50.45967031642573
170019696266969120 59.8864930019171 -50.66564304647051


## Lightcurves

In [16]:
result = L.object(objectId, lasair_added=False, reliabilityThreshold=0.5)
print15(json.dumps(result, indent=2))

{
  "diaObjectId": "170019696266969120",
  "diaSourcesList": [
    {
      "diaSourceId": 170019696266969120,
      "midpointMjdTai": 61088.098045202045,
      "band": "r",
      "psfFlux": 1244.3529052734375,
      "psfFluxErr": 219.41390991210938,
      "reliability": 0.34953588247299194
    },
    {
      "diaSourceId": 170032882351341601,
      "midpointMjdTai": 61091.03589422882,
      "band": "i",


## Sherlock Object

In [17]:
lite=True
result = L.sherlock_object(objectId, lite=lite)
print15(json.dumps(result, indent=2))

{
  "classifications": {
    "170019696266969120": [
      "SN",
      "The transient is possibly associated with <em>10000,74752,572</em>; a J=16.57 mag galaxy found in the DESI/2MASS catalogues. It's located 0.70\" S, 0.27\" W from the galaxy centre."
    ]
  },
  "crossmatches": [
    {
      "Mag": 17.358,
      "MagErr": 0.001,
      "MagFilter": "r",
      "association_type": "SN",
      "best_distance": null,
      "best_distance_flag": null,


## Sherlock Position

In [18]:
result = L.sherlock_position(ra, dec, lite=lite)
print15(json.dumps(result, indent=2))

{
  "classifications": {
    "query0": [
      "SN",
      "The transient is possibly associated with <em>03593276-5039549</em>; a J=16.57 mag galaxy found in the 2MASS/DESI catalogues. It's located 1.34\" S, 0.07\" W from the galaxy centre."
    ]
  },
  "crossmatches": [
    {
      "Mag": 17.358,
      "MagErr": 0.001,
      "MagFilter": "r",
      "association_type": "SN",
      "best_distance": null,
      "best_distance_flag": null,


## All object data

In [19]:
result = L.object(objectId, lasair_added=True)
print15(json.dumps(result, indent=2))

{
  "diaObjectId": "170019696266969120",
  "lasairData": {
    "nDiaSources": 3,
    "firstDiaSourceMjdTai": 61088.098045202045,
    "lastDiaSourceMjdTai": 61091.10058588956,
    "glat": -47.4948,
    "ebv": 0.00813893,
    "rasex": "03:59:32.758",
    "decsex": "-50:39:56.315",
    "ec_lon": 31.56692483820491,
    "ec_lat": -68.08458187694842,
    "g_lon": 259.58642769717454,
    "g_lat": -47.49478255970333,
    "now_mjd": "61091.57",


In [20]:
result = L.object(objectId, lasair_added=False)
print15(json.dumps(result, indent=2))

{
  "diaObjectId": "170019696266969120",
  "diaSourcesList": [
    {
      "diaSourceId": 170019696266969120,
      "midpointMjdTai": 61088.098045202045,
      "band": "r",
      "psfFlux": 1244.3529052734375,
      "psfFluxErr": 219.41390991210938,
      "reliability": 0.34953588247299194
    },
    {
      "diaSourceId": 170032882351341601,
      "midpointMjdTai": 61091.03589422882,
      "band": "i",
